In [8]:
import polars as pl
import pyprojroot

ROOT = pyprojroot.here()

In [ ]:
raw_data_temp1 = pl.read_parquet(ROOT/"data"/"raw"/"temporada1.parquet")
raw_data_temp2 = pl.read_parquet(ROOT/"data"/"raw"/"temporada2.parquet")

Saqué un par de variables de acá https://www.kaggle.com/code/thiagomantuani/predict-hits-with-new-mlb-get-started-here/notebook

In [ ]:
FT_TO_CM = 30.48
HALF_PLATE_CM = 21.59   # mitad de las 17 pulgadas del plato → 8.5 in * 2.54 cm/in
SHADOW_MARGIN_CM = 5.08  # 2 pulgadas fuera del borde de zona

swing_values = [
    "swinging_strike", "swinging_strike_blocked",
    "foul", "foul_tip", "foul_bunt", "bunt_foul_tip",
    "missed_bunt", "hit_into_play", "foul_pitchout"
]

cat_cols = [
    'batter', 'pitcher', 'stand', 'p_throws', 'pitch_type', 'count'
]

def aplicar_feature_engineering(df: pl.DataFrame, is_train: bool = True) -> pl.DataFrame:
    """
    Toma un DataFrame crudo, aplica las transformaciones originales
    y devuelve el dataset listo para modelar.
    """
    
    # 0. Conversión de unidades (pies a cm)
    df = df.with_columns( 
        plate_x = pl.col("plate_x") * FT_TO_CM,
        plate_z = pl.col("plate_z") * FT_TO_CM,
        pfx_x   = pl.col("pfx_x")   * FT_TO_CM,
        pfx_z   = pl.col("pfx_z")   * FT_TO_CM,
        sz_top  = pl.col("sz_top")  * FT_TO_CM, 
        sz_bot  = pl.col("sz_bot")  * FT_TO_CM
    )

    # 1. Respuesta (Solo se calcula para entrenamiento)
    if is_train and "description" in df.columns:
        df = df.with_columns(
            swing = pl.col("description").is_in(swing_values).cast(pl.Int8)  
        ).drop('description')

    # 2. Métricas base de zona y movimiento 
    df = df.with_columns( 
        sz_mid = (pl.col("sz_top") + pl.col("sz_bot")) / 2, # Centro vertical de la zona
        strike_zone_size = pl.col("sz_top") - pl.col("sz_bot"), # Altura total de la zona
        movement_complexity = (pl.col("pfx_x")**2 + pl.col("pfx_z")**2).sqrt(), # Magnitud total
        platoon_advantage = (pl.col("p_throws") != pl.col("stand")).cast(pl.Int8) # Ventaja
    )

    # 3. Localización normalizada
    df = df.with_columns(
        relative_height = ( # Vertical: 0 = piso de zona, 1 = techo
            (pl.col("plate_z") - pl.col("sz_bot")) / pl.col("strike_zone_size")
        ),
        relative_x = pl.col("plate_x") / HALF_PLATE_CM # Horizontal
    )

    # 4. Indicadoras de zona de strike
    df = df.with_columns(
        is_strike_zone = (
            pl.col("plate_x").is_between(-HALF_PLATE_CM, HALF_PLATE_CM) &
            pl.col("relative_height").is_between(0, 1)
        ).cast(pl.Int32)
    )
    
    df = df.with_columns(
        is_shadow_zone = (
            pl.col("plate_x").is_between(
                -HALF_PLATE_CM - SHADOW_MARGIN_CM,
                 HALF_PLATE_CM + SHADOW_MARGIN_CM
            ) &
            pl.col("relative_height").is_between(-0.1, 1.1) &
            (pl.col("is_strike_zone") == 0)
        ).cast(pl.Int32)
    )

    # 5. Distancia al centro de zona
    df = df.with_columns(
        pitch_location = (
            pl.col("plate_x")**2 +
            (pl.col("plate_z") - pl.col("sz_mid"))**2
        ).sqrt(),
    )

    # 6. Distancia al rincón más cercano (mide qué tan esquinado fue)
    df = df.with_columns(
        distance_to_corner = pl.when(pl.col("is_strike_zone") == 1).then(
            (
                (1 - pl.col("relative_x").abs())**2 +
                pl.min_horizontal(
                    pl.col("relative_height"),        # distancia al piso (0)
                    1 - pl.col("relative_height"),    # distancia al techo (1)
                )**2
            ).sqrt()
        ).otherwise(pl.lit(0.0))
    )

    # 7. Interacciones
    df = df.with_columns( 
        complex_speed = pl.col("release_speed") * pl.col("movement_complexity"),
        count = pl.col("balls").cast(pl.Utf8) + pl.lit("-") + pl.col("strikes").cast(pl.Utf8)
    )
    
    # FIX: Filtramos la categoría 1-3 para no romper XGBoost después
    df = df.with_columns(
        count = pl.when(pl.col("count") == "1-3").then(pl.lit(None)).otherwise(pl.col("count"))
    )

    # 8. Casteo de variables categóricas
    df = df.with_columns([
        pl.col(c).cast(pl.String).cast(pl.Categorical) for c in cat_cols if c in df.columns
    ])

    return df

t1_raw = pl.read_parquet(ROOT / "data" / "raw" / "temporada1.parquet")
t2_raw = pl.read_parquet(ROOT / "data" / "raw" / "temporada2.parquet")

print("Procesando Temporada 1...")
t1_procesada = aplicar_feature_engineering(t1_raw, is_train=True)

print("Procesando Temporada 2...")
t2_procesada = aplicar_feature_engineering(t2_raw, is_train=False)

# ==========================================
# GUARDADO DE ARCHIVOS
# ==========================================
print("Guardando archivos Parquet...")
path_procesados = ROOT / 'data' / 'processed'

# 1. Temporada 1 CON nulos (Para el XGBoost nuevo)
t1_procesada.write_parquet(path_procesados / 'temporada1_con_nulos.parquet')

# 2. Temporada 1 SIN nulos (Para tu compañero)
t1_procesada.drop_nulls().write_parquet(path_procesados / 'temporada1_limpio.parquet')

# 3. Temporada 2 lista para predecir
t2_procesada.write_parquet(path_procesados / 'temporada2_limpio.parquet')

print("¡Pipeline completado con éxito! Todos los archivos generados.")


data = (
    raw_data
    .with_columns( # 0. Conversión de unidades (pies a cm)
        plate_x = pl.col("plate_x") * FT_TO_CM,
        plate_z = pl.col("plate_z") * FT_TO_CM,
        pfx_x   = pl.col("pfx_x")   * FT_TO_CM,
        pfx_z   = pl.col("pfx_z")   * FT_TO_CM,
        sz_top  = pl.col("sz_top")  * FT_TO_CM,
        sz_bot  = pl.col("sz_bot")  * FT_TO_CM
    )
    .with_columns( # 1. Respuesta
        swing = pl.col("description").is_in(swing_values).cast(pl.Int8)  
    ).drop('description')
    .with_columns( # 2. Métricas base de zona y movimiento 
        sz_mid = (pl.col("sz_top") + pl.col("sz_bot")) / 2, # Centro vertical de la zona (varía por bateador)
        strike_zone_size = pl.col("sz_top") - pl.col("sz_bot"), # Altura total de la zona (pvaría por bateador)
        movement_complexity = (pl.col("pfx_x")**2 + pl.col("pfx_z")**2).sqrt(), # Magnitud total del movimiento del pitch 
        platoon_advantage = (pl.col("p_throws") != pl.col("stand")).cast(pl.Int8) # Ventaja de platoon: pitcher y bateador de lados opuestos
    )
    .with_columns( # 3. Localización normalizada
        relative_height = ( # Vertical: 0 = piso de zona, 1 = techo
            (pl.col("plate_z") - pl.col("sz_bot")) / pl.col("strike_zone_size")
        ),
        relative_x = pl.col("plate_x") / HALF_PLATE_CM # Horizontal: 0 = centro del plato, ±1 = bordes, >|1| = afuera
        # Simétrico: negativo = lado del bateador, positivo = lado contrario
    )
    .with_columns( # 4. Indicadoras de zona de strike
        # Dentro de zona reglamentaria
        is_strike_zone = (
            pl.col("plate_x").is_between(-HALF_PLATE_CM, HALF_PLATE_CM) &
            pl.col("relative_height").is_between(0, 1)
        ).cast(pl.Int32)
    )
    .with_columns(
        # Shadow zone: justo afuera de la zona (~2 pulgadas de margen)
        # Es donde los bateadores tienen mayor tasa de error de decisión
        is_shadow_zone = (
            pl.col("plate_x").is_between(
                -HALF_PLATE_CM - SHADOW_MARGIN_CM,
                 HALF_PLATE_CM + SHADOW_MARGIN_CM
            ) &
            pl.col("relative_height").is_between(-0.1, 1.1) &
            (pl.col("is_strike_zone") == 0)
        ).cast(pl.Int32)
    )
    .with_columns( #  5. Distancia al centro de zona
        pitch_location = (
            pl.col("plate_x")**2 +
            (pl.col("plate_z") - pl.col("sz_mid"))**2
        ).sqrt(),
    )
    .with_columns(# 6. Distancia al rincón más cercano (mide qué tan esquinado fue el lanzamiento)
    distance_to_corner = pl.when(pl.col("is_strike_zone") == 1).then(
        (
            (1 - pl.col("relative_x").abs())**2 +
            pl.min_horizontal(
                pl.col("relative_height"),        # distancia al piso (0)
                1 - pl.col("relative_height"),    # distancia al techo (1)
            )**2
        ).sqrt()
    ).otherwise(pl.lit(0.0))
    )
    .with_columns( # 7. Interacciones
        complex_speed = pl.col("release_speed") * pl.col("movement_complexity"), # Velocidad ponderada por movimiento total
        count = pl.col("balls").cast(pl.Utf8) + pl.lit("-") + pl.col("strikes").cast(pl.Utf8) # Cuenta de bolas y strikes
    )
    .drop_nulls()
)

In [5]:
data.columns

['pitch_id',
 'release_speed',
 'batter',
 'pitcher',
 'stand',
 'p_throws',
 'pitch_type',
 'balls',
 'strikes',
 'pfx_x',
 'pfx_z',
 'plate_x',
 'plate_z',
 'sz_top',
 'sz_bot',
 'swing',
 'sz_mid',
 'strike_zone_size',
 'movement_complexity',
 'platoon_advantage',
 'relative_height',
 'relative_x',
 'is_strike_zone',
 'is_shadow_zone',
 'pitch_location',
 'distance_to_corner',
 'complex_speed',
 'count']

In [6]:
cat_cols = [
 'batter',
 'pitcher',
 'stand',
 'p_throws',
 'pitch_type',
 'count'
 ]

data = data.with_columns([
    pl.col(c).cast(pl.String).cast(pl.Categorical) for c in cat_cols
])

In [7]:
data.write_parquet(file = ROOT / 'data' / 'processed' / 'temporada1_limpio.parquet')